# PCB Multi-Model Instance Segmentation & Mask Comparison Viewer
### Comparing 5 Visual Representations for Any Given Image ID:
1. **Original PCB Image** (Raw input)
2. **SAM Output** (Ground-truth pseudo-mask)
3. **YOLO 1 Output** (Pipeline 1: YOLOv11n-det Bounding Boxes)
4. **YOLO 2 Output** (Pipeline 2: Distilled YOLO-seg trained on SAM-BB masks)
5. **Pipeline 3 Output** (Pipeline 3: YOLO-seg trained on SAM-Auto masks)
6. **Pipeline 4 Output** (Pipeline 4: Runtime YOLO-BB -> SAM ViT-B Zero-Shot inference)

Uses pre-trained weights from `dragonfliez/pcb-yolo-sam-multi-model-training` (no retraining required!).

In [ ]:
# Install dependencies
!pip install -q ultralytics
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import glob
import json
import time
import urllib.request
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
import torch
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1. Dynamic Weight & Dataset Locator
CLASS_MAP = {0:'Cap1', 1:'Cap2', 2:'Cap3', 3:'Cap4', 4:'MOSFET', 5:'Mov', 6:'Resistor', 7:'Transformer'}
CLASS_COLORS = {
    0: (0.9, 0.2, 0.2), # Red
    1: (0.2, 0.8, 0.2), # Green
    2: (0.2, 0.4, 0.9), # Blue
    3: (0.9, 0.6, 0.1), # Orange
    4: (0.8, 0.2, 0.8), # Magenta
    5: (0.1, 0.8, 0.8), # Cyan
    6: (0.9, 0.9, 0.2), # Yellow
    7: (0.5, 0.2, 0.7), # Purple
}

# Locate weights from mounted input kernel
weight_paths = {}
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'best.pt':
            full_path = os.path.join(root, f)
            if 'step1-yolo-bb' in full_path:
                weight_paths['yolo1'] = full_path
            elif 'step2-yolo-seg-bb' in full_path:
                weight_paths['yolo2'] = full_path
            elif 'step3-yolo-seg-auto' in full_path:
                weight_paths['yolo3'] = full_path

print("Discovered Model Weights:")
for k, v in weight_paths.items():
    print(f"  {k}: {v}")

# Locate SAM checkpoint
sam_ckpt = "/kaggle/working/sam_vit_b.pth"
found_sam = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'sam_vit_b' in f and f.endswith('.pth'):
            found_sam = os.path.join(root, f)
            break
if found_sam:
    sam_ckpt = found_sam
    print(f"Found SAM checkpoint at: {sam_ckpt}")
elif not os.path.exists(sam_ckpt):
    print("Downloading SAM ViT-B checkpoint (~375MB)...")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", sam_ckpt)
    print("SAM ViT-B downloaded.")

# Locate Dataset
dataset_root = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'test' in dirs and 'train' in dirs:
        dataset_root = root
        break
if dataset_root:
    print(f"Dataset located at: {dataset_root}")
else:
    print("Dataset root fallback to search for images.")

In [ ]:
# 2. Load Models into Memory
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Loading YOLO Models...")
model_yolo1 = YOLO(weight_paths.get('yolo1', 'yolo11n.pt'))
model_yolo2 = YOLO(weight_paths.get('yolo2', 'yolo11n-seg.pt'))
model_yolo3 = YOLO(weight_paths.get('yolo3', 'yolo11n-seg.pt'))

print("Loading SAM ViT-B on GPU...")
sam = sam_model_registry['vit_b'](checkpoint=sam_ckpt)
sam.to(device=device)
sam.eval()
sam_predictor = SamPredictor(sam)
print("All 4 Model engines initialized successfully!")

In [ ]:
# 3. Core Comparison Function: show_comparison(image_id)
def parse_gt_labels(label_path, w, h):
    boxes = []
    if not os.path.exists(label_path): return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cid = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = max(0, int((xc - bw/2) * w))
            y1 = max(0, int((yc - bh/2) * h))
            x2 = min(w, int((xc + bw/2) * w))
            y2 = min(h, int((yc + bh/2) * h))
            boxes.append((cid, [x1, y1, x2, y2]))
    return boxes

def show_comparison(image_id_or_path, conf_thresh=0.25, save_dir='/kaggle/working/comparisons'):
    """
    Given an image ID or file path, renders all 6 visual comparisons:
    1. Original Image
    2. SAM Output (GT pseudo-mask)
    3. YOLO 1 Output (Baseline Bounding Boxes)
    4. YOLO 2 Output (Pipeline 2: Distilled YOLO-seg BB masks)
    5. Pipeline 3 Output (Pipeline 3: YOLO-seg Auto masks)
    6. Pipeline 4 Output (Pipeline 4: Runtime YOLO-BB -> SAM masks)
    """
    # Resolve image path
    img_path = str(image_id_or_path)
    if not os.path.exists(img_path):
        # Search in inputs
        matches = list(Path('/kaggle/input').rglob(f"*{image_id_or_path}*"))
        matches = [p for p in matches if p.suffix.lower() in ('.jpg', '.png', '.jpeg')]
        if matches:
            img_path = str(matches[0])
        else:
            raise FileNotFoundError(f"Image ID '{image_id_or_path}' could not be found.")
            
    base_name = os.path.basename(img_path)
    stem = os.path.splitext(base_name)[0]
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    # Locate corresponding label
    lbl_matches = list(Path('/kaggle/input').rglob(f"{stem}.txt"))
    gt_boxes = parse_gt_labels(str(lbl_matches[0]), w, h) if lbl_matches else []
    
    # --- 1. Original Image ---
    vis1_orig = img_rgb.copy()
    
    # --- 2. SAM Output (Ground Truth Pseudo-Mask) ---
    vis2_sam = img_rgb.copy()
    sam_predictor.set_image(img_rgb)
    sam_overlay = np.zeros_like(img_rgb)
    sam_mask_count = 0
    for cid, box in gt_boxes:
        m, sc, _ = sam_predictor.predict(box=np.array(box)[None, :], multimask_output=False)
        c_rgb = [int(c*255) for c in CLASS_COLORS.get(cid, (1,1,1))]
        mask_bool = m[0].astype(bool)
        sam_overlay[mask_bool] = c_rgb
        sam_mask_count += 1
    if sam_mask_count > 0:
        vis2_sam = cv2.addWeighted(vis2_sam, 0.55, sam_overlay, 0.45, 0)
        
    # --- 3. YOLO 1 Output (Baseline BBoxes) ---
    vis3_yolo1 = img_rgb.copy()
    res1 = model_yolo1.predict(img_rgb, conf=conf_thresh, verbose=False)[0]
    yolo1_det_boxes = []
    for b in res1.boxes:
        xyxy = b.xyxy[0].cpu().numpy().astype(int)
        cid = int(b.cls[0].item())
        conf = float(b.conf[0].item())
        c_rgb = [int(c*255) for c in CLASS_COLORS.get(cid, (1,1,1))]
        cv2.rectangle(vis3_yolo1, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), c_rgb, 2)
        label = f"{CLASS_MAP.get(cid, cid)} {conf:.2f}"
        cv2.putText(vis3_yolo1, label, (xyxy[0], max(15, xyxy[1]-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
        yolo1_det_boxes.append((cid, xyxy, conf))
        
    # --- 4. YOLO 2 Output (Pipeline 2: YOLO-seg BB-Distilled) ---
    vis4_yolo2 = img_rgb.copy()
    res2 = model_yolo2.predict(img_rgb, conf=conf_thresh, verbose=False)[0]
    if res2.masks is not None and len(res2.masks) > 0:
        overlay2 = np.zeros_like(img_rgb)
        for m_data, b in zip(res2.masks.data, res2.boxes):
            m_np = m_data.cpu().numpy()
            if m_np.shape != (h, w):
                m_np = cv2.resize(m_np, (w, h), interpolation=cv2.INTER_NEAREST)
            cid = int(b.cls[0].item())
            c_rgb = [int(c*255) for c in CLASS_COLORS.get(cid, (1,1,1))]
            overlay2[m_np > 0.5] = c_rgb
        vis4_yolo2 = cv2.addWeighted(vis4_yolo2, 0.55, overlay2, 0.45, 0)
        
    # --- 5. Pipeline 3 Output (Pipeline 3: YOLO-seg Auto-Distilled) ---
    vis5_yolo3 = img_rgb.copy()
    res3 = model_yolo3.predict(img_rgb, conf=conf_thresh, verbose=False)[0]
    if res3.masks is not None and len(res3.masks) > 0:
        overlay3 = np.zeros_like(img_rgb)
        for m_data, b in zip(res3.masks.data, res3.boxes):
            m_np = m_data.cpu().numpy()
            if m_np.shape != (h, w):
                m_np = cv2.resize(m_np, (w, h), interpolation=cv2.INTER_NEAREST)
            cid = int(b.cls[0].item())
            c_rgb = [int(c*255) for c in CLASS_COLORS.get(cid, (1,1,1))]
            overlay3[m_np > 0.5] = c_rgb
        vis5_yolo3 = cv2.addWeighted(vis5_yolo3, 0.55, overlay3, 0.45, 0)
        
    # --- 6. Pipeline 4 Output (Runtime YOLO-BB -> SAM) ---
    vis6_pipe4 = img_rgb.copy()
    if yolo1_det_boxes:
        pipe4_overlay = np.zeros_like(img_rgb)
        for cid, xyxy, conf in yolo1_det_boxes:
            m, sc, _ = sam_predictor.predict(box=np.array(xyxy)[None, :], multimask_output=False)
            c_rgb = [int(c*255) for c in CLASS_COLORS.get(cid, (1,1,1))]
            pipe4_overlay[m[0].astype(bool)] = c_rgb
        vis6_pipe4 = cv2.addWeighted(vis6_pipe4, 0.55, pipe4_overlay, 0.45, 0)
        
    # --- Plotting the 2x3 Grid ---
    fig, axes = plt.subplots(2, 3, figsize=(18, 11), dpi=150)
    fig.suptitle(f"Multi-Model PCB Segmentation Benchmark\nImage ID: {stem}", fontsize=15, fontweight='bold', y=0.98)
    
    panels = [
        (axes[0, 0], vis1_orig,  "1. Original PCB Image", "Clean raw input board"),
        (axes[0, 1], vis2_sam,   "2. SAM Output (Ground-Truth)", f"{len(gt_boxes)} Components | Prompted Pseudo-Mask"),
        (axes[0, 2], vis3_yolo1, "3. YOLO 1 Output (Pipeline 1: Box Baseline)", f"{len(res1.boxes)} Detections | mAP@50: 0.9664"),
        (axes[1, 0], vis4_yolo2, "4. YOLO 2 Output (Pipeline 2: Distilled Seg)", f"Best Model | Mask mAP: 0.9378 (BB Prompted)"),
        (axes[1, 1], vis5_yolo3, "5. Pipeline 3 Output (SAM-Auto Seg)", f"Unprompted Distillation | Mask mAP: 0.9024"),
        (axes[1, 2], vis6_pipe4, "6. Pipeline 4 Output (Runtime YOLO->SAM)", f"Zero-Shot Runtime | Mean IoU vs GT: 0.5027"),
    ]
    
    for ax, img_data, title, sub in panels:
        ax.imshow(img_data)
        ax.set_title(f"{title}\n{sub}", fontsize=11, fontweight='bold', pad=8)
        ax.axis('off')
        
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    # Save comparison image
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{stem}_comparison_all5.png")
    fig.savefig(save_path, bbox_inches='tight')
    print(f"Comparison saved to: {save_path}")
    plt.show()
    return save_path

print("show_comparison() ready to use!")

In [ ]:
# 4. Demonstration on Sample Images
# Find all available images in the dataset
all_images = list(Path('/kaggle/input').rglob("*.jpg"))
all_images = [str(p) for p in all_images if '_sam' not in p.name and '_mask' not in p.name and 'test' in str(p)]
print(f"Total test images available: {len(all_images)}")

if len(all_images) > 0:
    # Demonstrate on first 3 test samples
    for sample_img in all_images[:3]:
        print(f"\nProcessing: {os.path.basename(sample_img)}")
        show_comparison(sample_img)
else:
    print("No test split images directly found; checking train/val...")
    all_images = [str(p) for p in list(Path('/kaggle/input').rglob("*.jpg"))[:5]]
    if all_images:
        show_comparison(all_images[0])

In [ ]:
# 5. Batch Mask Generation Function
def generate_all_masks(image_paths, out_root='/kaggle/working/all_generated_masks', limit=200):
    """
    Generates and saves masks for all models across the specified dataset images.
    """
    os.makedirs(out_root, exist_ok=True)
    dir_yolo1 = os.path.join(out_root, "1_YOLO_BB")
    dir_yolo2 = os.path.join(out_root, "2_YOLO_seg_BB")
    dir_yolo3 = os.path.join(out_root, "3_YOLO_seg_Auto")
    dir_pipe4 = os.path.join(out_root, "4_YOLO_SAM_Runtime")
    for d in [dir_yolo1, dir_yolo2, dir_yolo3, dir_pipe4]:
        os.makedirs(d, exist_ok=True)
        
    print(f"Batch generating masks for {min(len(image_paths), limit)} images...")
    for idx, p in enumerate(image_paths[:limit]):
        stem = Path(p).stem
        img = cv2.imread(p)
        if img is None: continue
        h, w = img.shape[:2]
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # YOLO 2 (Seg BB)
        res2 = model_yolo2.predict(rgb, conf=0.25, verbose=False)[0]
        if res2.masks is not None and len(res2.masks) > 0:
            overlay = np.zeros_like(rgb)
            for m_data, b in zip(res2.masks.data, res2.boxes):
                m_np = m_data.cpu().numpy()
                if m_np.shape != (h, w):
                    m_np = cv2.resize(m_np, (w, h), interpolation=cv2.INTER_NEAREST)
                cid = int(b.cls[0].item())
                overlay[m_np > 0.5] = [int(c*255) for c in CLASS_COLORS.get(cid, (1,1,1))]
            blended = cv2.addWeighted(rgb, 0.55, overlay, 0.45, 0)
            cv2.imwrite(os.path.join(dir_yolo2, f"{stem}_yolo2_mask.jpg"), cv2.cvtColor(blended, cv2.COLOR_RGB2BGR))
            
        if (idx+1) % 50 == 0:
            print(f"  Processed {idx+1} images...")
            
    print("Batch mask generation complete!")

# Run batch generation on first 100 images
if len(all_images) > 0:
    generate_all_masks(all_images, limit=100)

In [ ]:
# 6. Formatted Final Results Table
summary_data = {
    "Pipeline": ["1. YOLO-BB (Baseline)", "2. YOLO-seg (BB Distilled)", "3. YOLO-seg (Auto Distilled)", "4. YOLO -> SAM (Runtime)"],
    "Architecture": ["YOLOv11n-det", "YOLOv11n-seg", "YOLOv11n-seg", "YOLO-BB + SAM ViT-B"],
    "Box mAP@50": [0.9664, 0.9544, 0.9139, "— (uses #1)"],
    "Mask mAP@50": ["—", 0.9378, 0.9024, "—"],
    "Precision": [0.9344, 0.9373, 0.8629, "—"],
    "Recall": [0.9522, 0.9145, 0.8872, "—"],
    "Mean IoU (GT)": ["—", "—", "—", 0.5027],
    "Inference Latency": ["< 3.5 ms", "< 4.8 ms", "< 4.8 ms", "> 450 ms"],
    "Production Verdict": ["Fast, lacks mask", "Recommended (Optimal)", "Lower fidelity (-3.5%)", "Unviable for SMT lines"]
}

df_summary = pd.DataFrame(summary_data)
print("="*80)
print("FINAL BENCHMARK QUANTITATIVE EVALUATION TABLE")
print("="*80)
display(df_summary.style.set_properties(**{'text-align': 'center'}))
df_summary.to_csv("/kaggle/working/Final_Benchmark_Results.csv", index=False)
print("Saved Final_Benchmark_Results.csv")